In [1]:
import os
import math
import time
import torch
import pickle
import numpy as np

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.autograd as autograd

import torch.nn.functional as functional
import torch.optim.lr_scheduler as lr_scheduler
from data_generation import *
from utils import *

os.makedirs('./results', exist_ok=True)
torch.set_default_dtype(torch.float64) 
_ = torch.manual_seed(0)

In [2]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
if torch.cuda.is_available():
    torch_device = 'cuda'
else:
    torch_device = 'cpu'
print(f"TORCH DEVICE: {torch_device}")

TORCH DEVICE: cuda


In [4]:
no_gaussians = 100
no_samples = 1000

In [5]:
# Multivariate gaussian
mean = [[0.,0.]]
for i in range(no_gaussians-1):
    mean.append([(2*i-1)/25.,(2*i+1)/25.])
mean = torch.tensor(mean)
cov  = torch.eye(2)

In [6]:
data = []
cond = []

for j in range(no_gaussians):
    dist = torch.distributions.MultivariateNormal(mean[j], covariance_matrix = cov,validate_args=False)
    x  = dist.sample((no_samples,))
    data.append(x)
    cond.append(mean[j].tile(no_samples,).reshape(-1,2))
    

In [8]:
batch_size = 100
data = torch.tensor(np.array(data),device = torch_device).reshape(-1,2)
cond = torch.tensor(np.array(cond),device = torch_device).reshape(-1,2)


In [ ]:
test_dataloader = torch.utils.data.DataLoader((data,cond), batch_size = batch_size, shuffle= False)

In [ ]:
def log_prob(x,mean, scale = 1.):
    logp = -1.0*torch.log(torch.tensor(2*np.pi)) - ((x[:, 0] - mean[:, 0]) ** 2 + (x[:, 1] - mean[:, 1]) ** 2)/(2 * scale**2)
    return logp

In [ ]:
# neg_logp = -log_prob(data,cond).cpu().detach().numpy()


In [ ]:
# neg_logp = []
# for x in test_dataloader:
#     print(x.shape)
#     (x1,x2) = x
#     print(x1.shape)
#     print(x2.shape)
#     logq = log_prob(x1,x2)
#     neg_logp.append(-logq.cpu().detach().numpy())

In [9]:
class SimpleNormal:
    def __init__(self, loc, var, device = torch_device):
        self.dist = torch.distributions.normal.Normal(torch.flatten(loc), torch.flatten(var))
        self.shape = loc.shape
        self.device = device
    def log_prob(self, x):
        logp = self.dist.log_prob(x.reshape(x.shape[0], -1)).to(self.device)
        return torch.sum(logp, dim=1)
    def sample_n(self, batch_size):
        x = self.dist.sample((batch_size,)).to(self.device)
        return x.reshape(batch_size, *self.shape)

In [10]:
class Affine_Coupling(nn.Module):
    def __init__(self,input_shape,cond_shape,hidden_dim):
        super(Affine_Coupling, self).__init__()
        self.input_dim = input_shape + cond_shape
        self.input_shape = input_shape
        self.hidden_dim = hidden_dim

        self.layer1 = nn.Linear(self.input_dim, self.hidden_dim)
        self.layer2 = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.scale = nn.Linear(self.hidden_dim, self.input_shape)
        self.translation = nn.Linear(self.hidden_dim, self.input_shape)


    def _compute(self, x):
        out = torch.relu(self.layer1(x))
        out = torch.relu(self.layer2(out))
        return out
    
    def forward(self, x):
        ## convert latent space variable to observed variable
        out = self._compute(x)
        s   = torch.tanh(self.scale(out))
        t   = self.translation(out)
        
        return s,t


In [11]:
class RNVP_2D(nn.Module):
    '''
    A RealNVP class for modeling 2 dimensional distributions
    '''
    def __init__(self,num_coupling_layers,input_shape,cond_shape,hidden_dim):
        '''
        initialized with a list of masks. each mask define an affine coupling layer
        '''
        super(RNVP_2D, self).__init__()        
        self.hidden_dim = hidden_dim  
        self.num_coupling_layers = num_coupling_layers
        self.distribution = SimpleNormal(torch.zeros((2,),device = torch_device), torch.ones((2,),device = torch_device), device = torch_device)
        self.layers_list = nn.ModuleList([Affine_Coupling(input_shape,cond_shape,hidden_dim) for i in range(num_coupling_layers)])
        self.masks = torch.tensor(np.array([[0, 1], [1, 0]] * (num_coupling_layers // 2),
                                                            dtype="float32"),requires_grad = False,device = torch_device)
        

    def forward(self, x):
        ## convert latent space variables into observed variables
        (x1,x2) = x
        ldj  = 0
        logq = self.distribution.log_prob(x1)
        for i in range(self.num_coupling_layers):
            s,t  = self.layers_list[i](torch.cat((x1*self.masks[i] , x2),dim = -1))
            y1   = self.masks[i]*x1 + (1-self.masks[i])*(x1*torch.exp(s) + t)  
            y2   = x2
            ldj += torch.sum((1 - self.masks[i])*s, -1)
            x1   = y1
        logq -= ldj
        y = (y1,y2)
        return y, logq

    def inverse(self, y):
        ## convert observed variables into latent space variables    
        (y1,y2) = y
        ldj = 0
        for i in reversed(range(self.num_coupling_layers)):
            s,t = self.layers_list[i](torch.cat((y1*self.masks[i] , y2),dim = -1))
            x1 = self.masks[i]*y1 + (1-self.masks[i])*((y1 - t)*torch.exp(-s))
            x2 = y2
            ldj += torch.sum((1 - self.masks[i])*(-s), -1)
            y1 = x1
        logq  = self.distribution.log_prob(x1)
        logq += ldj
        x = (x1,x2)
        return x, logq

    def log_pdf(self,y):
        (y1,y2) = y
        ldj = 0
        for i in reversed(range(self.num_coupling_layers)):
            s,t = self.layers_list[i](torch.cat((y1*self.masks[i] , y2),dim = -1))
            x1 = self.masks[i]*y1 + (1-self.masks[i])*((y1 - t)*torch.exp(-s))
            x2 = y2
            ldj += torch.sum((1 - self.masks[i])*(-s), -1)
            y1 = x1
        logq  = self.distribution.log_prob(x1)
        logq += ldj
        return logq
        
    
    def jacobian(self, x):
        logq = self.log_pdf(x)
        v = torch.zeros_like(logq)
        v[:] = 1.
        dy_dx = autograd.grad(logq, x, grad_outputs=v, retain_graph=True,
                                create_graph=True, allow_unused=True)[0]  # shape [B, N]
        
        jacobian = dy_dx.requires_grad_()
        return jacobian
      
    
    
        
 

In [12]:
model = RNVP_2D(8,(2),(2),256).to(torch_device)
optimizer = optim.Adam(model.parameters(),lr= 1.e-5)
PATH = "./results/model_fkl.pt"

In [13]:
#For resuming the checkpoint:
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [14]:
model.eval()
neg_logp = - model.log_pdf((data,cond)).cpu().detach().numpy()

In [15]:
data_with_mu = torch.cat((data,cond),dim = -1).cpu().detach().numpy()

In [16]:
data_with_negp = [data_with_mu,neg_logp]

In [17]:
output = open("Training_data_multivariate_multi_mean_unit_cov_and_neg_logp_trained_with_nf.pkl", 'wb')
pickle.dump(data_with_negp, output)
output.close()